[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week02_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 2 Assignment: Real-World A/B Analysis & Monitoring Metrics

## QuickCart — Building an Experiment Health Dashboard

QuickCart's experimentation team has just launched a test on their **recommendation engine**. The new algorithm (pilot) is expected to increase average order value by surfacing higher-relevance products.

Your job: build a set of **daily monitoring metrics** that let the team track the experiment’s health in real time. These metrics will power an internal dashboard that alerts on anomalies and helps the team decide when to stop or escalate an experiment.

You will work with raw transaction-level data and produce clean daily aggregates for both the pilot and control groups.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

## Synthetic Data

Below we generate a transaction dataset that mimics QuickCart’s order records. Each row is a single line item: one product within one sale (order).

In [ ]:
np.random.seed(42)

# Parameters
n_users = 1000
n_days = 14
start_date = '2020-01-01'
dates = pd.date_range(start_date, periods=n_days, freq='D')

# Assign users to groups
user_ids = np.arange(1, n_users + 1)
pilot_users = user_ids[:500]
control_users = user_ids[500:]

# Generate transactions
rows = []
sale_id = 1000

for day in dates:
    # Each user has ~30% chance of ordering on any day
    active_users = np.random.choice(user_ids, size=int(n_users * 0.3), replace=False)
    for uid in active_users:
        sale_id += 1
        # Each order has 1-5 items
        n_items = np.random.randint(1, 6)
        for _ in range(n_items):
            cost = np.round(np.random.exponential(25) + 5, 2)
            rows.append({
                'date': day,
                'user_id': uid,
                'sale_id': sale_id,
                'cost': cost
            })

# Inject an anomaly on day 8: double the orders from a small set of users
anomaly_date = dates[7]
anomaly_users = np.random.choice(user_ids[:100], size=50, replace=False)
for uid in anomaly_users:
    sale_id += 1
    n_items = np.random.randint(3, 8)
    for _ in range(n_items):
        cost = np.round(np.random.exponential(80) + 20, 2)
        rows.append({
            'date': anomaly_date,
            'user_id': uid,
            'sale_id': sale_id,
            'cost': cost
        })

df_transactions = pd.DataFrame(rows)
df_transactions['date'] = pd.to_datetime(df_transactions['date'])

print(f"Dataset shape: {df_transactions.shape}")
print(f"Date range: {df_transactions['date'].min()} to {df_transactions['date'].max()}")
print(f"Unique users: {df_transactions['user_id'].nunique()}")
print(f"Unique sales: {df_transactions['sale_id'].nunique()}")
df_transactions.head(10)

---

## Task 1: Implement `calculate_sales_metrics`

### Problem

Write a function that takes the raw transaction DataFrame and computes **daily aggregate metrics** over a specified period. The function must:

1. **Filter by date**: only include rows where `begin <= date < end`
2. **Filter by users** (optional): if `filters` is provided, keep only rows matching the filter values
3. **Compute four metrics per day**:
   - `revenue`: total sum of the cost column for that day
   - `number_purchases`: number of **unique sale_ids** for that day
   - `average_check`: mean of the *total cost per sale_id* for that day (i.e., group by sale_id, sum costs, then take the mean)
   - `average_number_items`: mean number of *items per sale_id* for that day (i.e., group by sale_id, count rows, then take the mean)
4. **Return all dates** in the period as index, filling missing days with 0

### Function signature

```python
def calculate_sales_metrics(df, cost_name, date_name, sale_id_name, period, filters=None):
    """
    Parameters
    ----------
    df : pd.DataFrame
        Raw transaction data (one row per line item).
    cost_name : str
        Column name for item cost.
    date_name : str
        Column name for the date.
    sale_id_name : str
        Column name for the unique order/sale identifier.
    period : dict
        {'begin': '2020-01-01', 'end': '2020-01-08'}
        begin is inclusive, end is exclusive.
    filters : dict or None
        e.g., {'user_id': [111, 123]} — keep only rows where user_id is in the list.
        If None, use all rows.
    
    Returns
    -------
    pd.DataFrame
        Index = dates (daily), columns = ['revenue', 'number_purchases', 'average_check', 'average_number_items']
    """
```

<details>
<summary>Hint 1: Date filtering (click to expand)</summary>

```python
mask = (df[date_name] >= period['begin']) & (df[date_name] < period['end'])
```

Make sure dates are parsed as datetime before comparing.

</details>

<details>
<summary>Hint 2: Applying optional filters</summary>

```python
if filters is not None:
    for col, values in filters.items():
        mask &= df[col].isin(values)
```

</details>

<details>
<summary>Hint 3: Computing average_check</summary>

For each day, first group by `sale_id_name` and sum costs to get the total per order. Then take the mean of those totals.

```python
day_data.groupby(sale_id_name)[cost_name].sum().mean()
```

</details>

<details>
<summary>Hint 4: Filling missing dates</summary>

Create a full date range with `pd.date_range`, then `reindex` your result and fill NaN with 0.

</details>

In [ ]:
def calculate_sales_metrics(df, cost_name, date_name, sale_id_name, period, filters=None):
    """Compute daily sales metrics from transaction data.
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw transaction data (one row per line item).
    cost_name : str
        Column name for item cost.
    date_name : str
        Column name for the date.
    sale_id_name : str
        Column name for the unique order/sale identifier.
    period : dict
        {'begin': '2020-01-01', 'end': '2020-01-08'}
        begin is inclusive, end is exclusive.
    filters : dict or None
        e.g., {'user_id': [111, 123]} -- keep only matching rows.
        If None, use all rows.
    
    Returns
    -------
    pd.DataFrame
        Index = dates, columns = ['revenue', 'number_purchases', 'average_check', 'average_number_items']
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Test Cases for Task 1 ---

# Test with known small data
test_df = pd.DataFrame({
    'date': pd.to_datetime(['2020-01-01', '2020-01-01', '2020-01-01',
                            '2020-01-02', '2020-01-02',
                            '2020-01-03']),
    'user_id': [1, 1, 2, 2, 3, 1],
    'sale_id': [100, 100, 101, 102, 102, 103],
    'cost': [10.0, 20.0, 15.0, 30.0, 25.0, 40.0]
})

period_test = {'begin': '2020-01-01', 'end': '2020-01-04'}
result = calculate_sales_metrics(test_df, 'cost', 'date', 'sale_id', period_test)

print("Result:")
print(result)
print()

# Day 1: sale_id 100 (items: 10+20=30, 2 items), sale_id 101 (15, 1 item)
#   revenue = 10+20+15 = 45
#   number_purchases = 2 (sale_ids 100, 101)
#   average_check = (30 + 15) / 2 = 22.5
#   average_number_items = (2 + 1) / 2 = 1.5
assert abs(result.loc[pd.Timestamp('2020-01-01'), 'revenue'] - 45.0) < 0.01, "Day 1 revenue incorrect"
assert result.loc[pd.Timestamp('2020-01-01'), 'number_purchases'] == 2, "Day 1 purchases incorrect"
assert abs(result.loc[pd.Timestamp('2020-01-01'), 'average_check'] - 22.5) < 0.01, "Day 1 avg check incorrect"
assert abs(result.loc[pd.Timestamp('2020-01-01'), 'average_number_items'] - 1.5) < 0.01, "Day 1 avg items incorrect"

# Day 2: sale_id 102 (items: 30+25=55, 2 items)
#   revenue = 55
#   number_purchases = 1
#   average_check = 55
#   average_number_items = 2
assert abs(result.loc[pd.Timestamp('2020-01-02'), 'revenue'] - 55.0) < 0.01, "Day 2 revenue incorrect"
assert result.loc[pd.Timestamp('2020-01-02'), 'number_purchases'] == 1, "Day 2 purchases incorrect"
assert abs(result.loc[pd.Timestamp('2020-01-02'), 'average_check'] - 55.0) < 0.01, "Day 2 avg check incorrect"
assert abs(result.loc[pd.Timestamp('2020-01-02'), 'average_number_items'] - 2.0) < 0.01, "Day 2 avg items incorrect"

# Day 3: sale_id 103 (40, 1 item)
assert abs(result.loc[pd.Timestamp('2020-01-03'), 'revenue'] - 40.0) < 0.01, "Day 3 revenue incorrect"

print("All basic tests passed!")

In [ ]:
# Test: date filtering (end is exclusive)
period_short = {'begin': '2020-01-01', 'end': '2020-01-02'}
result_short = calculate_sales_metrics(test_df, 'cost', 'date', 'sale_id', period_short)
assert len(result_short) == 1, "Period should only include 1 day"
assert pd.Timestamp('2020-01-02') not in result_short.index, "End date should be exclusive"
print("Date filtering test passed!")

# Test: user filter
result_filtered = calculate_sales_metrics(
    test_df, 'cost', 'date', 'sale_id', period_test,
    filters={'user_id': [1]}
)
# User 1 has: Day 1 (sale 100: 10+20), Day 3 (sale 103: 40)
assert abs(result_filtered.loc[pd.Timestamp('2020-01-01'), 'revenue'] - 30.0) < 0.01, "Filtered revenue incorrect"
assert result_filtered.loc[pd.Timestamp('2020-01-02'), 'revenue'] == 0, "Day 2 should be 0 for user 1"
print("User filter test passed!")

# Test: missing dates filled with 0
period_gap = {'begin': '2020-01-01', 'end': '2020-01-06'}
result_gap = calculate_sales_metrics(test_df, 'cost', 'date', 'sale_id', period_gap)
assert len(result_gap) == 5, f"Should have 5 days, got {len(result_gap)}"
assert result_gap.loc[pd.Timestamp('2020-01-04'), 'revenue'] == 0, "Missing dates should be 0"
print("Missing date fill test passed!")

print("\nAll Task 1 tests passed!")

---

## Task 2: Compare Pilot vs Control Group Metrics

### Problem

Use your `calculate_sales_metrics` function to compute daily metrics separately for:
- **Pilot group**: users 1–500 (`pilot_users`)
- **Control group**: users 501–1000 (`control_users`)

Then create a 2x2 grid of plots comparing the two groups across all four metrics over the full 14-day period.

<details>
<summary>Hint: Structuring the comparison</summary>

1. Call `calculate_sales_metrics` twice with different `filters` arguments
2. Use `plt.subplots(2, 2, figsize=(14, 8))` for the grid
3. Plot both groups on each subplot with a legend

</details>

In [ ]:
# YOUR CODE HERE
# 1. Compute metrics for pilot group (users 1-500)
# 2. Compute metrics for control group (users 501-1000)
# 3. Plot a 2x2 grid comparing the groups

period_full = {'begin': '2020-01-01', 'end': '2020-01-15'}

# metrics_pilot = calculate_sales_metrics(...)
# metrics_control = calculate_sales_metrics(...)

---

## Task 3: Anomaly Detection in Daily Metrics

### Problem

Looking at the metrics you computed, identify days where the values are **anomalous**. An anomaly is defined as a day where a metric is more than **2 standard deviations** away from the mean of the other days.

Steps:
1. For each metric, compute the mean and standard deviation (excluding the day being tested — leave-one-out)
2. Flag any day where `|value - mean| > 2 * std`
3. Report which days are flagged and for which metrics

A simpler approach (acceptable): compute overall mean and std, then flag outliers.

<details>
<summary>Hint: Simple z-score approach</summary>

```python
for col in metrics.columns:
    z_scores = (metrics[col] - metrics[col].mean()) / metrics[col].std()
    anomalies = metrics[z_scores.abs() > 2]
```

</details>

<details>
<summary>Hint: What to look for in the data</summary>

Recall that the data generation code injected an anomaly on a specific day. Your anomaly detection should catch this.

</details>

In [ ]:
# YOUR CODE HERE
# 1. Compute metrics for all users over the full period
# 2. For each metric, flag anomalous days (|z-score| > 2)
# 3. Print or display the flagged days

**Your analysis:**

*Which day(s) were flagged? Which metrics were affected? Is this consistent with what you see in the plots from Task 2?*

---

## Bonus: Dashboard Summary Table

Create a summary table showing for each metric:
- Mean (pilot vs control)
- Relative difference (%)
- Whether the difference is statistically significant (use a t-test on daily values)

This is what a real experiment dashboard would display.

In [ ]:
# YOUR CODE HERE (optional)